# Archived historical CORAL experiment

> Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.


In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import timm
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
import seaborn as sn
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import sys
# Historical machine-specific sys.path removed from archived notebook.
from pyDeepInsightv3 import ImageTransformer

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder


In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
import gc
import torch

# Delete any leftover GPU tensors/models from previous notebook sessions
for obj_name in list(globals().keys()):
    obj = globals()[obj_name]
    if isinstance(obj, torch.Tensor) and obj.is_cuda:
        del globals()[obj_name]
    elif isinstance(obj, torch.nn.Module):
        obj.cpu()

gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory — allocated: {allocated:.2f} GiB | reserved: {reserved:.2f} GiB | total: {total:.2f} GiB")

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
import gc
import torch

# Delete any leftover GPU tensors/models from previous notebook sessions
for obj_name in list(globals().keys()):
    obj = globals()[obj_name]
    if isinstance(obj, torch.Tensor) and obj.is_cuda:
        del globals()[obj_name]
    elif isinstance(obj, torch.nn.Module):
        obj.cpu()

gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory — allocated: {allocated:.2f} GiB | reserved: {reserved:.2f} GiB | total: {total:.2f} GiB")

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
#import dataset
#SOURCE
expr_file1= r"C:\\Zmyfiles\codes\research\KPCA-DA\KPCA-DA\examples\Data\cptacvssmc\cptac_labels.csv"
expr1 = pd.read_csv(expr_file1, encoding='utf-8', engine='python')

#TARGET
expr_file2= r"C:\\Zmyfiles\codes\research\KPCA-DA\KPCA-DA\examples\Data\cptacvssmc\smc_labels.csv"
expr2 = pd.read_csv(expr_file2, encoding='utf-8', engine='python')


In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
#y_labels
Y1 = expr1.T.iloc[1:,-1].values
le = LabelEncoder()
y_train_enc = le.fit_transform(Y1)

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")

#y_labels
Y2 = expr2.T.iloc[1:,-1].values
le = LabelEncoder()
y_train_enc2 = le.fit_transform(Y2)

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
Y2.shape

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
expr1.set_index('Hugo_Symbol',inplace=True)
expr2.set_index('Hugo_Symbol',inplace=True)

genes1 = (expr1.index).tolist()
genes2 = (expr2.index).tolist()


In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
common_genes = sorted(list(set(genes1).intersection(set(genes2))))
if len(common_genes) == 0:
        raise ValueError("No common genes found between source and target datasets.")

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
# Find shared genes between source and target

# Subset to shared genes
X_source = (expr1.loc[common_genes].values).T  # shape: (n_source_samples, n_shared_genes)
X_target = (expr2.loc[common_genes].values).T  # shape: (n_target_samples, n_shared_genes)

#y_source = source_labels.values

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
X_source.shape, X_target.shape

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
# CORAL needs to invert a (K x K) covariance matrix via scipy.sqrtm,
# which casts to complex128: K=17484 → 4.56 GiB → MemoryError.
# Keep only the top-K most variable genes (computed on combined data).
K = 5000   # covariance matrix = 2000×2000×16 bytes ≈ 64 MB — manageable

combined_var = np.var(np.vstack([X_source, X_target]).astype(float), axis=0)
top_k_idx    = np.argsort(combined_var)[-K:]

X_source = X_source[:, top_k_idx].astype(float)
X_target = X_target[:, top_k_idx].astype(float)

print(f"Kept top {K} most variable genes")
print(f"X_source: {X_source.shape},  X_target: {X_target.shape}")

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
from sklearn.model_selection import train_test_split

X_target_adapt, X_target_test,Y_adapt,Y_test = train_test_split(
    X_target, 
    y_train_enc2,
    test_size=0.30, 
    random_state=42
)

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
from adapt.feature_based import CORAL
from sklearn.dummy import DummyClassifier

# CORAL aligns source distribution to match target.
# Xt = target (SMC adapt set) — the reference distribution.
# fit(X_source, y_source) — the data being aligned.
coral = CORAL(
    estimator=DummyClassifier(),
    Xt=X_target_adapt,
    lambda_=1e-5,
    verbose=1
)

coral.fit(X_source, y_train_enc)

X_source_aligned       = coral.transform(X_source)
X_target_test_aligned  = coral.transform(X_target_test)

print(f"X_source_aligned shape      : {X_source_aligned.shape}")
print(f"X_target_test_aligned shape : {X_target_test_aligned.shape}")

### DeepInsight and Classification

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
from sklearn.decomposition import PCA, KernelPCA


In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
# X_train, X_test, y_train, y_test = train_test_split(
#       X, Y, test_size=0.3, random_state=5, stratify=Y)

X_train, X_test, y_train, y_test = X_source_aligned ,X_target_test_aligned ,y_train_enc,Y_test

# var_filter1 = X_train.var(0) >= np.percentile(X_train.var(0), 10., method='nearest')
# var_filter2 = X_test.var(0) >= np.percentile(X_test.var(0), 10., method='nearest')

# mms = MinMaxScaler()
# X_train_norm = mms.fit_transform(X_train[:, var_filter1])
# X_test_norm = mms.fit_transform(X_test[:, var_filter2])

# le = LabelEncoder()
# y_train_enc = le.fit_transform(y_train)
# y_test_enc = le.transform(y_test)

# le_mapping = dict(zip(le.transform(le.classes_), le.classes_))
num_classes = np.unique(y_train_enc).size

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
import gc
import torch

# Delete any leftover GPU tensors/models from previous notebook sessions
for obj_name in list(globals().keys()):
    obj = globals()[obj_name]
    if isinstance(obj, torch.Tensor) and obj.is_cuda:
        del globals()[obj_name]
    elif isinstance(obj, torch.nn.Module):
        obj.cpu()

gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory — allocated: {allocated:.2f} GiB | reserved: {reserved:.2f} GiB | total: {total:.2f} GiB")

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = timm.create_model('resnet50d', pretrained=True, num_classes=num_classes).to(device)
net = net.to(device)

batch_size=10
preprocess = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Normalize(mean=net.default_cfg['mean'], std=net.default_cfg['std']),
])



In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    net.parameters(), 
    lr=1e-04,
    momentum=0.9,
)

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
reducer=KernelPCA(n_components=2, kernel='rbf',gamma=.2)

pixel_size = (227,227)

it1 = ImageTransformer(
    feature_extractor=reducer,
    pixels=pixel_size)

it2 = ImageTransformer(
    feature_extractor=reducer,
    pixels=pixel_size)

it1.fit(X_train, y=y_train, plot=False)
it2.fit(X_test, y=y_test, plot=False)

X_train_img = it1.transform(X_train)
X_test_img = it2.transform(X_test)

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")

#defing the data loader
X_train_tensor = torch.stack([preprocess(img) for img in X_train_img]).float().to(device)
y_train_tensor = torch.from_numpy(y_train_enc).long().to(device)
trainset = TensorDataset(X_train_tensor, y_train_tensor)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True)

X_test_tensor = torch.stack([preprocess(img) for img in X_test_img]).float().to(device)
y_test_tensor = torch.from_numpy(y_test).long().to(device)
testset = TensorDataset(X_test_tensor, y_test_tensor)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False)


##training the data using resnet:
net.train()
for epoch in range(50):
    for i, (inputs, labels) in enumerate(trainloader):
        clear_output(wait=True)
        #display(f"Epoch {epoch:03d}, batch {i:03d}")
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        

###testing the data
test_predicted = np.empty(0)
test_true = np.empty(0)
#test_pred = np.empty(0)

net.eval()

with torch.no_grad():
    for i, (inputs, labels) in enumerate(testloader):
        pred = torch.max(net(inputs), dim=1)[1].cpu().detach().numpy()
        test_predicted = np.append(test_predicted, pred)
        test_true = np.append(test_true, labels.cpu().detach().numpy())


#test_pred = (test_predicted > 0.5).astype(int)        

net=net.to(torch.device('cpu'))
with torch.no_grad():
    y_hat = net(X_train_tensor.cpu())
train_predicted=torch.max(y_hat,1)[1]
 

print(f"The train accuracy was {accuracy_score(train_predicted, y_train_enc):.3f}")
print(f"The test accuracy was {accuracy_score(test_predicted, test_true):.3f}")

        

In [ ]:
raise RuntimeError("Archived historical CORAL notebook: it selects/splits target data at sample level, uses target labels in the evaluation workflow, and fits a separate target image map. Its results are not valid evidence for the paper's held-out patient protocol. Use a patient-grouped comparator with frozen preprocessing and a source-selected classifier.")
test_true, test_predicted